# Test 01 — Mostly correct solution
This submission implements the compute_monthly_stats function closely following the reference. Intended mistake pattern: mostly correct (should get high score).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# Load data
p = Path.cwd() / 'data' / 'GW05_original_monthly.csv'
data_monthly = pd.read_csv(p, sep=';', decimal=',')
data_monthly['date'] = pd.to_datetime(data_monthly['yyyymm'], format='%Y%m')
data_monthly = data_monthly[data_monthly['date'] >= '1927-12-01'].copy()
data_monthly.set_index('date', inplace=True)
# predictors
data_monthly['dp'] = np.log(data_monthly['D12']) - np.log(data_monthly['Index'])
data_monthly['dy'] = np.log(data_monthly['D12']) - np.log(data_monthly['Index'].shift(1))
data_monthly['ep'] = np.log(data_monthly['E12']) - np.log(data_monthly['Index'])
data_monthly['de'] = np.log(data_monthly['D12']) - np.log(data_monthly['E12'])
data_monthly['e10p'] = np.log(data_monthly['E12'].rolling(window=120).mean()) - np.log(data_monthly['Index'])
data_monthly['tms'] = data_monthly['AAA'] - data_monthly['tbl']
data_monthly['dfy'] = data_monthly['BAA'] - data_monthly['AAA']
data_monthly['equity_premium'] = data_monthly['CRSP_SPvw'] - data_monthly['Rfree']

def compute_monthly_stats(ts_df, indep, start, end, est_periods_OOS=240):
    dep = 'equity_premium'
    ts = ts_df.loc[pd.to_datetime(start):pd.to_datetime(end)].copy()
    def lagged_x(df, var):
        return df[var].shift(2) if var == 'infl' else df[var].shift(1)
    # in-sample
    x = lagged_x(ts, indep)
    idx = x.dropna().index
    y = ts.loc[idx, dep]
    X = add_constant(x.loc[idx])
    reg = OLS(y, X).fit()
    T = len(y)
    k = int(reg.df_resid) + 1
    r2 = reg.rsquared
    is_r2_head = r2 - (1 - r2) * (T - k) / (T - 1)
    # OOS rolling
    idx_all = ts_df.index
    start_pos = idx_all.searchsorted(pd.to_datetime(start), side='left')
    end_pos = idx_all.searchsorted(pd.to_datetime(end), side='right') - 1
    OOS_error_M, OOS_error_C = [], []
    for pos in range(start_pos + est_periods_OOS, end_pos):
        actual = ts_df.iloc[pos + 1][dep]
        avg_i = ts_df.iloc[start_pos:pos + 1][dep].mean()
        OOS_error_M.append(actual - avg_i)
        ts_train = ts_df.iloc[start_pos:pos + 1]
        x_train = lagged_x(ts_train, indep)
        idx_train = x_train.dropna().index
        y_train = ts_train.loc[idx_train, dep]
        X_train = add_constant(x_train.loc[idx_train])
        reg_oos = OLS(y_train, X_train).fit()
        x_new = ts_df.iloc[pos][indep]
        x_new_const = add_constant(pd.DataFrame({indep: [x_new]}), has_constant='add')
        pred = reg_oos.predict(x_new_const)[0]
        OOS_error_C.append(pred - actual)
    mse_m = np.mean(np.array(OOS_error_M) ** 2)
    OOS_R2_C = 1 - (np.mean(np.array(OOS_error_C) ** 2) / mse_m)
    Tn = len(idx_train)
    kn = Tn - 1
    OOS_R2_head = OOS_R2_C - (1 - OOS_R2_C) * (Tn - kn) / (Tn - 1)
    return {'IS_R2_head': round(float(is_r2_head) * 100, 2), 'OOS_R2_head': round(float(OOS_R2_head) * 100, 2)}

vars_list = ['dp','dy','ep','de','e10p','tms','dfy']
results = {}
for var in vars_list:
    results[var] = compute_monthly_stats(data_monthly, var, '1927-12-01','2005-12-01', est_periods_OOS=240)
df_results = pd.DataFrame.from_dict(results, orient='index')
df_results.index.name = 'variable'
df_results